In [31]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# Load and prep data
df = pd.read_csv('processed_ai4i2020.csv')
features = ['Tool wear [min]', 'Temp_diff', 'Rotational speed [rpm]', 'Power', 'Tool_Torque', 'Type']
X = df[features]
y = df['Machine failure']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Step 1: Default model results
print("=== Default Model Results ===")
default_models = {
    'XGBoost': XGBClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(random_state=42),
    'AdaBoost': AdaBoostClassifier(random_state=42),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'SVM': SVC(random_state=42, probability=True),
    'KNN': KNeighborsClassifier()
}

default_results = {}
for name, model in default_models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, 'predict_proba') else None
    default_results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob) if y_prob is not None else 'N/A'
    }
print(pd.DataFrame(default_results).T)

# Step 2: Parameter tuning
print("\n=== Parameter Tuning ===")
scale_pos = (len(y_train) - sum(y_train)) / sum(y_train)  # For XGBoost

tuning_models = {
    'XGBoost': {
        'model': XGBClassifier(random_state=42, scale_pos_weight=scale_pos),
        'params': {'n_estimators': [474], 'max_depth': [10], 'learning_rate': [0.06], 'subsample': [0.8], 'colsample_bytree': [1.0], 'gamma': [0.01], 'reg_alpha': [0.01], 'reg_lambda': [0]}
    },
    'RandomForest': {
        'model': RandomForestClassifier(random_state=42, class_weight='balanced'),
        'params': {'n_estimators': [47], 'max_depth': [None], 'min_samples_split': [3], 'min_samples_leaf': [1], 'max_features': ['sqrt'], 'bootstrap': [True]}
    },
    'AdaBoost': {
        'model': AdaBoostClassifier(random_state=42),
        'params': {'n_estimators': [248], 'learning_rate': [1.6]}
    },
    'DecisionTree': {
        'model': DecisionTreeClassifier(random_state=42, class_weight='balanced'),
        'params': {'max_depth': [24], 'min_samples_split': [2], 'min_samples_leaf': [1], 'criterion': ['entropy'], 'max_leaf_nodes': [None], 'min_impurity_decrease': [0.0]}
    },
    'SVM': {
        'model': SVC(random_state=42, probability=True, class_weight='balanced'),
        'params': {'C': [50], 'gamma': [1], 'kernel': ['rbf'], 'degree': [2], 'coef0': [0.0]}
    },
    'KNN': {
        'model': KNeighborsClassifier(),
        'params': {'n_neighbors': [3], 'weights': ['uniform'], 'metric': ['manhattan'], 'p': [1], 'algorithm': ['auto']}
    }
}

best_models = {}
for name, info in tuning_models.items():
    grid = GridSearchCV(info['model'], info['params'], cv=5, scoring='f1', n_jobs=-1)
    grid.fit(X_train_scaled, y_train)
    best_models[name] = grid.best_estimator_
    print(f"{name} Best Params: {grid.best_params_}")

# Step 3: Rerun with tuned models
print("\n=== Tuned Model Results ===")
tuned_results = {}
for name, model in best_models.items():
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, 'predict_proba') else None
    tuned_results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob) if y_prob is not None else 'N/A'
    }
print(pd.DataFrame(tuned_results).T)

=== Default Model Results ===
              Accuracy        F1   ROC-AUC
XGBoost         0.9900  0.827586  0.967966
RandomForest    0.9925  0.867257  0.943388
AdaBoost        0.9815  0.574713  0.981451
DecisionTree    0.9845  0.755906  0.888801
SVM             0.9815  0.602151  0.952705
KNN             0.9810  0.604167  0.865386

=== Parameter Tuning ===
XGBoost Best Params: {'colsample_bytree': 1.0, 'gamma': 0.01, 'learning_rate': 0.06, 'max_depth': 10, 'n_estimators': 474, 'reg_alpha': 0.01, 'reg_lambda': 0, 'subsample': 0.8}
RandomForest Best Params: {'bootstrap': True, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 3, 'n_estimators': 47}
AdaBoost Best Params: {'learning_rate': 1.6, 'n_estimators': 248}
DecisionTree Best Params: {'criterion': 'entropy', 'max_depth': 24, 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2}
SVM Best Params: {'C': 50, 'coef0': 0.0, 'degree': 2, 'gamma': 1, 'kernel':